# VisionPort – Sign Language Recognition
## Skill 3: Request Pipeline, Sequence Diagram, Execution and Latency Analysis

**Same project as Skill 2:** static ASL alphabet recognition using the Sign Language MNIST model (24 classes; J and Z are excluded because they require movement).

### What this notebook demonstrates
1. **Activity 1:** request validation, image decoding, resizing, tensor preparation, model invocation, postprocessing and JSON response.
2. **Activity 2:** sequence diagram for one request, including data format and tensor shape at each stage.
3. **Activity 3:** execute a prediction, inspect outputs and test invalid requests.
4. **Activity 4:** measure approximate time spent in preprocessing, inference and response preparation.

> This notebook runs inference locally with PyTorch so the complete request can be demonstrated in Colab. It does not pretend that a Triton server is running. A separate optional section shows where Triton fits if you have a reachable Triton deployment.

## Before you start

From your **Skill 2 notebook**, download the trained weights file named `sign_language_cnn.pth` from `/content/visionport_artifacts/`. In Colab, you can run this in Skill 2:

```python
from google.colab import files
files.download("/content/visionport_artifacts/sign_language_cnn.pth")
```

Then upload that `.pth` file when prompted in this notebook. You do **not** need to train the model again.

## Cell 1 – Check runtime and install required packages

In [ ]:
import sys, platform, torch
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

Python: 3.13.18
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
PyTorch: 2.14.1+cpu
CUDA available: False
Using: cpu


## Cell 2 – Install API and image-processing packages

In [ ]:
!pip -q install fastapi uvicorn python-multipart pillow requests nest-asyncio

## Cell 3 – Upload the trained Skill 2 model weights

In [ ]:
from google.colab import files
uploaded_model = files.upload()  # Choose sign_language_cnn.pth
MODEL_PATH = next(iter(uploaded_model))
print("Model file:", MODEL_PATH)

## Cell 4 – Recreate the exact CNN architecture from Skill 2 and load weights

In [ ]:
import torch
import torch.nn as nn
import os

LETTERS = [c for c in "ABCDEFGHIJKLMNOPQRSTUVWXYZ" if c not in ("J", "Z")]

class SignLanguageCNN(nn.Module):
    def __init__(self, num_classes=24):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((3, 3))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Quick sanity check on the uploaded file extension to prevent loading images
_, ext = os.path.splitext(MODEL_PATH.lower())
if ext in (".jpg", ".jpeg", ".png"):
    raise ValueError(f"Error: The file '{MODEL_PATH}' appears to be an image. Please re-run Cell 3 and upload your model weights file (e.g., 'sign_language_cnn.pth').")

model = SignLanguageCNN(num_classes=len(LETTERS)).to(DEVICE)
state = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state)
model.eval()
print("Loaded trained CNN successfully.")
print("Number of output classes:", len(LETTERS))

## Cell 5 – Upload an ASL image for one prediction request

In [ ]:
from google.colab import files
image_upload = files.upload()  # Choose a hand-sign image
IMAGE_PATH = next(iter(image_upload))
print("Image received:", IMAGE_PATH)

## Cell 6 – Activity 1: implement the full request pipeline

The function below performs the same stages that an API request needs:
- validate that bytes were supplied and the image can be decoded;
- decode the image;
- convert to grayscale and resize to 28×28;
- normalize pixel values to 0–1;
- create a tensor with shape **(1, 1, 28, 28)**;
- invoke the trained CNN;
- convert scores to probabilities and select the most likely letter;
- prepare a JSON-compatible response.

**Preprocessing note:** The model was trained on Sign Language MNIST-style grayscale images. A normal phone photo may not match the training images (background, hand crop, lighting and orientation), so an incorrect prediction is possible even when the code runs correctly.

In [ ]:
from PIL import Image, UnidentifiedImageError
import io, numpy as np, time, json

def run_inference_request(image_bytes, filename="uploaded_image"):
    timings = {}
    total_start = time.perf_counter()

    # 1. Request validation
    t = time.perf_counter()
    if not image_bytes:
        raise ValueError("No image data received.")
    timings["validation_ms"] = (time.perf_counter() - t) * 1000

    # 2. Image decoding
    t = time.perf_counter()
    try:
        image = Image.open(io.BytesIO(image_bytes))
        image.load()
    except (UnidentifiedImageError, OSError):
        raise ValueError("Invalid image file. Upload a PNG or JPG image.")
    original_size = image.size
    timings["decoding_ms"] = (time.perf_counter() - t) * 1000

    # 3. Resize and grayscale
    t = time.perf_counter()
    gray = image.convert("L")
    resized = gray.resize((28, 28), Image.Resampling.BILINEAR)
    array = np.asarray(resized, dtype=np.float32) / 255.0
    timings["resize_preprocess_ms"] = (time.perf_counter() - t) * 1000

    # 4. Tensor preparation: (batch, channel, height, width)
    t = time.perf_counter()
    tensor = torch.from_numpy(array).unsqueeze(0).unsqueeze(0).to(DEVICE)
    tensor_shape = list(tensor.shape)
    timings["tensor_prepare_ms"] = (time.perf_counter() - t) * 1000

    # 5. Model invocation
    t = time.perf_counter()
    with torch.inference_mode():
        logits = model(tensor)
    timings["inference_ms"] = (time.perf_counter() - t) * 1000

    # 6. Postprocessing
    t = time.perf_counter()
    probabilities = torch.softmax(logits, dim=1)[0]
    class_index = int(torch.argmax(probabilities).item())
    predicted_letter = LETTERS[class_index]
    confidence = float(probabilities[class_index].item())
    timings["postprocess_ms"] = (time.perf_counter() - t) * 1000

    # 7. Response serialization
    response = {
        "filename": filename,
        "predicted_sign": predicted_letter,
        "confidence": round(confidence, 4),
        "status": "success"
    }
    timings["total_ms"] = (time.perf_counter() - total_start) * 1000

    trace = {
        "original_image_size": list(original_size),
        "decoded_format": image.format,
        "resized_format": "grayscale 28x28",
        "tensor_dtype": str(tensor.dtype),
        "tensor_shape": tensor_shape,
        "model_output_shape": list(logits.shape),
        "response": response,
        "timings": timings
    }
    return response, trace, resized

## Cell 7 – Run the uploaded image through the pipeline

In [ ]:
with open(IMAGE_PATH, "rb") as f:
    image_bytes = f.read()

response_data, trace_data, processed_image = run_inference_request(
    image_bytes, filename=IMAGE_PATH
)

display(processed_image)
print("API-style JSON response:")
print(json.dumps(response_data, indent=2))
print("\nRequest trace:")
print(json.dumps(trace_data, indent=2))

## Activity 2 – Sequence diagram and data-format trace

This diagram represents **one request** from upload to response. The tensor shape shown is the shape passed into the CNN. Triton can replace the local model invocation stage in a deployed version.

```mermaid
sequenceDiagram
    actor User
    participant API as FastAPI
    participant Pre as Image Preprocessing
    participant Model as CNN Model (local in this notebook)
    User->>API: POST /predict + image file
    API->>API: Validate upload
    API->>Pre: Image bytes
    Pre->>Pre: Decode image
    Pre->>Pre: Grayscale + resize 28x28
    Pre->>Pre: Normalize + create tensor
    Note over Pre,Model: FP32 tensor shape (1, 1, 28, 28)
    Pre->>Model: Input tensor
    Model->>Model: CNN inference
    Note over Model: Output scores shape (1, 24)
    Model-->>API: 24 class scores
    API->>API: Softmax + argmax + letter mapping
    API-->>User: JSON prediction + confidence
```

### Data format and tensor shape at each stage

In [ ]:
stage_trace = [
    ("HTTP upload", "multipart/form-data file", "image bytes"),
    ("Validation", "bytes + filename", "non-empty bytes"),
    ("Image decoding", "JPEG/PNG bytes", f"PIL image {trace_data['original_image_size']}"),
    ("Grayscale + resize", "PIL image", "28 x 28, 1 channel"),
    ("Normalization", "uint8 pixels 0..255", "float32 array 28 x 28, values 0..1"),
    ("Tensor preparation", "float32 array", str(trace_data["tensor_shape"])),
    ("CNN invocation", "FP32 tensor", str(trace_data["model_output_shape"]) + " output scores"),
    ("Postprocessing", "24 scores", "softmax probabilities -> argmax -> letter"),
    ("Response", "Python dictionary", "JSON object")
]
for stage, data_format, shape in stage_trace:
    print(f"{stage:24} | {data_format:30} | {shape}")

## Cell 8 – Activity 3: create a FastAPI endpoint

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse

app = FastAPI(
    title="VisionPort Sign Language Recognition – Skill 3",
    description="Demonstrates validation, preprocessing, CNN inference and JSON response.",
    version="1.0"
)

@app.get("/")
def home():
    return {"project": "VisionPort Sign Language Recognition", "status": "running"}

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if not file.filename:
        raise HTTPException(status_code=400, detail="Filename is required.")
    contents = await file.read()
    try:
        result, trace, _ = run_inference_request(contents, filename=file.filename)
        return result
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))

print("FastAPI app is ready. Endpoint: POST /predict")

## Cell 9 – Start the API inside Colab

In [ ]:
import nest_asyncio, threading, uvicorn, time, requests
nest_asyncio.apply()

API_PORT = 8001
config = uvicorn.Config(app, host="127.0.0.1", port=API_PORT, log_level="warning")
server = uvicorn.Server(config)
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()

for _ in range(30):
    try:
        health = requests.get(f"http://127.0.0.1:{API_PORT}/", timeout=1)
        if health.ok:
            break
    except requests.RequestException:
        time.sleep(1)

print("API status:", health.status_code, health.json())
print("Interactive API documentation is available at /docs on the running server.")

## Cell 10 – Test the actual HTTP request and JSON response

In [ ]:
with open(IMAGE_PATH, "rb") as f:
    api_response = requests.post(
        f"http://127.0.0.1:{API_PORT}/predict",
        files={"file": (IMAGE_PATH, f, "image/jpeg")},
        timeout=60
    )

print("HTTP status:", api_response.status_code)
print("Response JSON:")
print(json.dumps(api_response.json(), indent=2))

## Cell 11 – Test request validation with an invalid file

In [ ]:
bad_response = requests.post(
    f"http://127.0.0.1:{API_PORT}/predict",
    files={"file": ("not_an_image.txt", b"this is not an image", "text/plain")},
    timeout=30
)
print("HTTP status for invalid upload:", bad_response.status_code)
print("Validation response:", bad_response.json())

## Cell 12 – Activity 4: inspect latency points

In [ ]:
print("Measured time for this request (milliseconds):")
for stage, ms in trace_data["timings"].items():
    print(f"{stage:24}: {ms:.3f} ms")

print("\nLatency interpretation:")
print("- Upload/network time is not measured by the local function timer.")
print("- Decoding and resizing depend on image size and CPU.")
print("- Inference depends on model size and CPU/GPU availability.")
print("- API/network overhead is outside the internal stage timings.")

## Optional: where Triton and TensorRT fit

In the fully deployed Skill 2 architecture, the local call:

```python
logits = model(tensor)
```

is replaced by a Triton inference request. FastAPI sends the FP32 tensor to Triton; Triton loads the exported ONNX or TensorRT model and returns the scores. TensorRT is an optimization/runtime option for compatible NVIDIA hardware.

**Do not claim Triton or TensorRT was executed in this notebook.** This notebook demonstrates the request pipeline locally with the same trained CNN. To demonstrate a real Triton request, a reachable Triton server and deployed model repository are required.

## Viva preparation

1. **What is request validation?** Checking that the request contains a usable image before processing it.
2. **Why decode an image?** The uploaded bytes must be converted into pixel data.
3. **Why resize to 28×28?** The CNN was trained with 28×28 grayscale images.
4. **What is a tensor?** A multidimensional numerical array used as model input.
5. **What does shape (1, 1, 28, 28) mean?** Batch size 1, one grayscale channel, height 28 and width 28.
6. **Why does the model output (1, 24)?** It produces one score for each of the 24 supported letters.
7. **What is postprocessing?** Converting scores into probabilities and selecting the highest-scoring class.
8. **What is response serialization?** Returning the result in a structured format such as JSON.
9. **Where can latency occur?** Upload, decoding, preprocessing, data transfer, inference, postprocessing and response delivery.
10. **Why might a phone photo be misclassified?** It may differ from the training images in crop, background, lighting, hand position or orientation.

### Completion checklist
- [ ] Uploaded Skill 2 `.pth` weights
- [ ] Uploaded an ASL image
- [ ] Ran Activity 1 pipeline
- [ ] Recorded input and output tensor shapes for Activity 2
- [ ] Reviewed the sequence diagram
- [ ] Tested FastAPI `/predict`
- [ ] Tested invalid image validation
- [ ] Recorded latency measurements
- [ ] Saved screenshots of outputs for review